# 05 - Avaliacao Comparativa e Explicabilidade (XAI)
Comparamos os dois modelos, analisamos o limiar de decisao e usamos
**Permutation Importance** e **SHAP** para entender o modelo.

In [ ]:
# bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.options.display.float_format = '{:.3f}'.format
import sys; sys.path.append("../src")   # acesso a preprocessing.py
DATA = "../data/processed"


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num = ["idade_anos", "NU_CONTATO", "dias_diag_trat"]
cat = ["CS_SEXO","CS_RACA","CS_ESCOL_N","SG_UF_NOT","RAIOX_TORA","TESTE_TUBE",
       "BACILOSC_E","HIV","AGRAVAIDS","AGRAVALCOO","AGRAVDIABE","AGRAVDOENC",
       "AGRAVDROGA","AGRAVTABAC","TRAT_SUPER","ANT_RETRO","BENEF_GOV",
       "POP_RUA","POP_LIBER","POP_IMIG","POP_SAUDE"]

def preparar(df):
    # cria o atraso ate o inicio do tratamento e deixa as categoricas como texto
    df = df.copy()
    df["DT_DIAG"]    = pd.to_datetime(df["DT_DIAG"], errors="coerce")
    df["DT_INIC_TR"] = pd.to_datetime(df["DT_INIC_TR"], errors="coerce")
    df["dias_diag_trat"] = (df["DT_INIC_TR"] - df["DT_DIAG"]).dt.days
    for c in cat:
        df[c] = df[c].fillna("ignorado").astype(str)
    return df[num + cat]
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score
# para a explicabilidade usamos a base completa de treino
treino = pd.read_csv(f"{DATA}/treino.csv")  # base completa
teste1 = pd.read_csv(f"{DATA}/teste1.csv")

## 1. Comparacao dos modelos

In [ ]:
resumo = pd.DataFrame({
    "acuracia":[0.693, 0.680, 0.710, 0.699],
    "recall":  [0.758, 0.783, 0.769, 0.756],
    "f1":      [0.684, 0.682, 0.786, 0.777],
    "roc_auc": [0.781, 0.785, 0.752, 0.757],
}, index=["LogReg-Teste1","RedeNeural-Teste1","LogReg-Teste2","RedeNeural-Teste2"])
resumo

**Conclusao:** a rede neural supera o baseline em ROC-AUC e F1 nos dois testes,
mantendo recall alto (prioridade clinica).

## 2. Analise do limiar de decisao (threshold)

In [ ]:
# treina um modelo so pra essa analise
transf = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat),
], verbose_feature_names_out=False)
Xtr = transf.fit_transform(preparar(treino)); ytr = treino["ltfu"]
modelo = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, ytr)
proba = modelo.predict_proba(transf.transform(preparar(teste1)))[:,1]; y = teste1["ltfu"]

linhas = []
for t in np.arange(0, 1.01, 0.05):
    p = (proba >= t).astype(int)
    linhas.append([t, accuracy_score(y,p), precision_score(y,p,zero_division=0), recall_score(y,p,zero_division=0)])
tab = pd.DataFrame(linhas, columns=["threshold","acuracia","precisao","recall"])
plt.plot(tab["threshold"], tab["precisao"], label="Precisao")
plt.plot(tab["threshold"], tab["recall"], label="Recall")
plt.xlabel("Limiar"); plt.ylabel("Score"); plt.legend(); plt.grid(True)
plt.title("Precisao e Recall por limiar de decisao"); plt.show()

Reduzir o limiar **aumenta o recall** (captura mais abandonos) ao custo de
precisao - util quando perder um caso de abandono e mais grave que um alarme falso.

## 3. Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance
# embaralha cada variavel e ve quanto o ROC-AUC cai
Xa = transf.transform(preparar(teste1))
r = permutation_importance(modelo, Xa, teste1["ltfu"], scoring="roc_auc", n_repeats=5, random_state=42)
imp = pd.Series(r.importances_mean, index=transf.get_feature_names_out()).sort_values()
imp.tail(10).plot(kind="barh", color="#4c72b0")
plt.xlabel("Queda no ROC-AUC ao embaralhar"); plt.title("Variaveis mais importantes"); plt.show()

## 4. SHAP

In [ ]:
import shap
# SHAP para modelo linear (LinearExplainer e rapido)
Xs = transf.transform(preparar(treino.sample(800, random_state=0)))
explainer = shap.LinearExplainer(modelo, Xs)
shap.summary_plot(explainer.shap_values(Xs), features=Xs,
                  feature_names=list(transf.get_feature_names_out()), max_display=12, show=False)
plt.tight_layout(); plt.show()

## 5. Sanity check: pacientes ficticios

In [ ]:
# teste de sanidade: compara um perfil de alto risco com um de baixo
alto = treino.iloc[0].copy()
alto["AGRAVALCOO"]="1"; alto["AGRAVDROGA"]="1"; alto["TRAT_SUPER"]="2"; alto["HIV"]="1"; alto["idade_anos"]=28
baixo = treino.iloc[0].copy()
baixo["AGRAVALCOO"]="2"; baixo["AGRAVDROGA"]="2"; baixo["TRAT_SUPER"]="1"; baixo["HIV"]="2"; baixo["idade_anos"]=55
casos = pd.DataFrame([alto, baixo])
prob = modelo.predict_proba(transf.transform(preparar(casos)))[:,1]
print(f"Paciente ALTO risco  (alcool+drogas+sem DOT+HIV): {prob[0]:.1%}")
print(f"Paciente BAIXO risco (sem agravos, com DOT)     : {prob[1]:.1%}")


O modelo responde de forma **coerente**: maior probabilidade para o perfil de risco.